# To-do lists

`TodoListMiddleware` adds `write_todos` and stores task status in agent state. Planning does not execute work by itself; watch the separate handbook and calculator calls.

[Task planning](https://docs.langchain.com/oss/python/langchain/middleware/built-in#to-do-list)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.tools import add, search_handbook
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, TodoListMiddleware
from model_config import get_model, run_config
from lab.faults import ScriptedModel, tool_request
from langchain_core.messages import AIMessage


In [ ]:
print(prompt("todo"))

In [ ]:
# Execute
agent = create_agent(
    model=get_model(), tools=[search_handbook, add],
    middleware=[TodoListMiddleware(), ModelCallLimitMiddleware(run_limit=10), TraceMiddleware("07_todo", pretty=True)],
)
result = agent.invoke({"messages": [("user", prompt("todo"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)
print("Final TODO state:", result.get("todos"))

### Replace and complete the plan

`write_todos` replaces the current task list; statuses are `pending`, `in_progress` and `completed`. It records progress but does not execute tasks. Custom `system_prompt` and `tool_description` values control planning instructions and the tool guidance.

In [ ]:
# Execute
plan_agent = create_agent(model=ScriptedModel(responses=[
    tool_request("write_todos", todos=[{"content": "Read policy", "status": "in_progress"}, {"content": "Summarize policy", "status": "pending"}]),
    tool_request("write_todos", todos=[{"content": "Read policy", "status": "completed"}, {"content": "Summarize policy", "status": "completed"}]),
    AIMessage(content="The plan is complete."),
]), middleware=[TodoListMiddleware(system_prompt="Track progress with write_todos and finish every task.", tool_description="Replace the plan and update each task's status.")])
plan_result = plan_agent.invoke({"messages": [("user", "Track the two tasks.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(plan_result["messages"], pretty=True)
print("Plan:", plan_result["todos"])
assert all(todo["status"] == "completed" for todo in plan_result["todos"])